# 📚 Notebook 01 — 市场数据基础

**第一阶段：基础知识** · 前置要求：基础 Python

---

## 🎯 学习目标

完成本 Notebook 后，你将能够：

1. 解释什么是 **OHLCV** 数据，并读懂K线图
2. 使用 Python 从 Binance 公共 API **获取历史K线数据**
3. 构建一个**结构化数据类** (`BinanceKline`) 来管理金融记录
4. 处理 **API 分页**以检索长期历史数据（每页 1000 行限制）
5. 将K线数据**存储和查询**到本地 SQLite 数据库
6. 用 matplotlib 绘制专业的**K线图**

In [ ]:
# ── 环境设置（每次会话运行一次）──
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print("✅ 环境设置完成")

---

## 📐 第一节：什么是 OHLCV 数据？

所有交易系统都从**价格数据**开始。通用格式是 **OHLCV** — 五个数字概括一个时间段内所有交易活动：

| 字段 | 含义 | 示例 |
|---|---|---|
| **O**pen（开盘价） | 时间段开始时的价格 | $67,450.00 |
| **H**igh（最高价） | 时间段内的最高价格 | $68,200.00 |
| **L**ow（最低价） | 时间段内的最低价格 | $67,100.00 |
| **C**lose（收盘价） | 时间段结束时的价格 | $67,800.00 |
| **V**olume（交易量） | 时间段内的总交易量 | 1,234.56 BTC |

### K线（蜡烛图）结构

```
     ┃       ← 上影线（最高价）
   ┌─┴─┐
   │   │     ← 实体（开盘 → 收盘）
   │   │        绿色：收盘 > 开盘（价格上涨）
   │   │        红色：收盘 < 开盘（价格下跌）
   └─┬─┘
     ┃       ← 下影线（最低价）
```

每根K线代表一个**时间间隔**（1分钟、1小时、1天等）。

### 为什么使用 OHLCV？

- **紧凑**：将数千笔独立交易压缩为5个数字
- **通用**：每个交易所、每种资产类别都使用此格式
- **充分**：大多数量化策略只需要 OHLCV 数据
- **成交量很重要**：成交量确认价格变动 — 高成交量突破更可靠

---

## 💻 第二节：从 Binance 获取数据

Binance 提供一个**免费公共 API** 用于获取历史K线数据。无需 API 密钥。

### API 接口

```
GET https://api.binance.com/api/v3/klines
    ?symbol=BTCUSDT
    &interval=1h
    &limit=1000
    &startTime=<毫秒时间戳>
    &endTime=<毫秒时间戳>
```

每行返回 **13 个值**，以 JSON 数组形式：

| 索引 | 字段 | 类型 |
|---|---|---|
| 0 | 开盘时间（毫秒） | int |
| 1 | 开盘价 | string |
| 2 | 最高价 | string |
| 3 | 最低价 | string |
| 4 | 收盘价 | string |
| 5 | 成交量（基础资产） | string |
| 6 | 收盘时间（毫秒） | int |
| 7 | 报价资产成交量 | string |
| 8 | 成交笔数 | int |
| 9 | 主动买入基础资产量 | string |
| 10 | 主动买入报价资产量 | string |

让我们获取一些数据：

In [ ]:
# ── 原始 API 调用：获取 100 根 BTC 小时K线 ──
import requests

url = "https://api.binance.com/api/v3/klines"
params = {
    "symbol": "BTCUSDT",
    "interval": "1h",
    "limit": 100,
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()
raw_klines = response.json()

print(f"获取了 {len(raw_klines)} 根K线")
print(f"第一根K线（原始数据）: {raw_klines[0][:5]}...")  # 显示前5个字段

### 问题：原始数据很混乱

API 返回的是**混合类型的数组** — 时间戳是整数，价格是字符串，成交量是字符串。我们需要一个干净的、有类型的结构。

### 解决方案：`BinanceKline` 数据类

我们的生产代码使用了一个 **frozen dataclass** — 一个不可变的命名记录，具有正确的类型。让我们从零开始构建一个，然后与生产版本对比。

In [ ]:
# ── 从零构建 BinanceKline 数据类 ──
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True, slots=True)
class Kline:
    """来自 Binance 的标准化 OHLCV 记录。
    
    frozen=True  → 不可变（防止意外修改数据）
    slots=True   → 内存高效（没有 __dict__）
    """
    symbol: str
    interval: str
    open_time_ms: int      # Unix 时间戳（毫秒）
    close_time_ms: int
    open: float
    high: float
    low: float
    close: float
    volume: float          # 基础资产成交量（如 BTC）
    quote_volume: float    # 报价资产成交量（如 USDT）
    trade_count: int

    @classmethod
    def from_api_row(cls, symbol: str, interval: str, row: list[Any]) -> "Kline":
        """将一行原始 API 数据解析为有类型的 Kline 记录。"""
        return cls(
            symbol=symbol,
            interval=interval,
            open_time_ms=int(row[0]),
            open=float(row[1]),
            high=float(row[2]),
            low=float(row[3]),
            close=float(row[4]),
            volume=float(row[5]),
            close_time_ms=int(row[6]),
            quote_volume=float(row[7]),
            trade_count=int(row[8]),
        )

# 将原始数据解析为有类型的记录
klines = [Kline.from_api_row("BTCUSDT", "1h", row) for row in raw_klines]

print(f"\n解析了 {len(klines)} 根K线")
print(f"第一根: open={klines[0].open:,.2f}, close={klines[0].close:,.2f}, volume={klines[0].volume:,.4f}")
print(f"最后一根: open={klines[-1].open:,.2f}, close={klines[-1].close:,.2f}")

### 💻 与生产代码对比

我们简化的 `Kline` 与 `bot/data/binance_fetcher.py` 中的生产版 `BinanceKline` 非常接近。生产版还包括：

- `taker_buy_base_volume` 和 `taker_buy_quote_volume`（主动买入跟踪）
- `normalize_binance_symbol()` 处理交易对格式差异（如 `BTCUSD` → `BTCUSDT`）
- 与 `tenacity` 重试库集成，实现弹性 API 调用

让我们看看生产代码的获取器：

In [ ]:
# ── 生产代码中的 BinanceFetcher ──
from bot.data.binance_fetcher import BinanceFetcher, BinanceKline, normalize_binance_symbol

# 生产代码获取器的关键特性：
print("交易对标准化示例：")
print(f"  BTCUSD  → {normalize_binance_symbol('BTCUSD')}")
print(f"  ETHUSDT → {normalize_binance_symbol('ETHUSDT')}")
print(f"  SOL/USD → {normalize_binance_symbol('SOL/USD')}")

print(f"\n支持的时间间隔: {list(BinanceFetcher.interval_to_milliseconds.__func__.__code__.co_consts)}")

# 生产代码获取器使用以下机制包装 requests：
# - @retry(stop=stop_after_attempt(3), wait=wait_exponential(...))
# - 处理 HTTP 429（频率限制）和 5xx（服务器错误）作为可重试错误
# - 对不可重试的失败抛出 BinanceApiError

---

## 📐 第三节：API 分页 — 获取长期历史数据

Binance 限制每次请求最多 **1000 行**。要获取 90 天的小时数据，我们需要：

$$\text{总K线数} = 90 \times 24 = 2{,}160 \quad \Rightarrow \quad \lceil 2{,}160 / 1{,}000 \rceil = 3 \text{ 页}$$

### 分页策略

```
第1页: startTime = T₀         → 获取 1000 根K线 → last_open_time = T₉₉₉
第2页: startTime = T₉₉₉ + 1h → 获取 1000 根K线 → last_open_time = T₁₉₉₉
第3页: startTime = T₁₉₉₉ + 1h → 获取 160 根K线  → 完成（不足 1000）
```

关键思路：每页完成后，将 `startTime` 推进到 `last_open_time + interval_ms`。

In [ ]:
# ── 从零构建分页获取器 ──

def fetch_all_klines(
    symbol: str,
    interval: str = "1h",
    days: int = 30,
    limit: int = 1000,
) -> list[Kline]:
    """通过逐页调用 Binance API 获取指定交易对的所有K线数据。"""
    # 计算时间范围
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(days=days)
    
    # 转换为毫秒（Binance 使用毫秒时间戳）
    cursor_ms = int(start_time.timestamp() * 1000)
    end_ms = int(end_time.timestamp() * 1000)
    
    # 间隔宽度（毫秒），用于推进游标
    interval_ms = {"1h": 3_600_000, "1d": 86_400_000, "4h": 14_400_000}[interval]
    
    all_klines: list[Kline] = []
    page = 0
    
    while cursor_ms < end_ms:
        page += 1
        response = requests.get(
            "https://api.binance.com/api/v3/klines",
            params={
                "symbol": symbol,
                "interval": interval,
                "startTime": cursor_ms,
                "endTime": end_ms,
                "limit": limit,
            },
            timeout=10,
        )
        response.raise_for_status()
        rows = response.json()
        
        if not rows:
            break
        
        page_klines = [Kline.from_api_row(symbol, interval, r) for r in rows]
        all_klines.extend(page_klines)
        
        # 将游标推进到最后一根K线之后
        last_open = page_klines[-1].open_time_ms
        cursor_ms = last_open + interval_ms
        
        print(f"  第 {page} 页: 获取了 {len(rows)} 根K线（累计: {len(all_klines)}）")
        
        if len(rows) < limit:
            break  # 最后一页
    
    return all_klines

# 获取 30 天的 BTC 小时K线
print("正在获取 BTCUSDT 小时K线 (30天)...")
btc_klines = fetch_all_klines("BTCUSDT", "1h", days=30)
print(f"\n✅ 共获取: {len(btc_klines)} 根K线")

---

## 📊 第四节：可视化K线数据

In [ ]:
# ── 转换为 DataFrame 用于分析和绘图 ──

def klines_to_dataframe(klines: list[Kline]) -> pd.DataFrame:
    """将 Kline 列表转换为 pandas DataFrame。"""
    records = [
        {
            "timestamp": pd.Timestamp(k.open_time_ms, unit="ms", tz="UTC"),
            "open": k.open,
            "high": k.high,
            "low": k.low,
            "close": k.close,
            "volume": k.volume,
            "quote_volume": k.quote_volume,
            "trade_count": k.trade_count,
        }
        for k in klines
    ]
    df = pd.DataFrame(records).set_index("timestamp")
    return df

btc_df = klines_to_dataframe(btc_klines)
print(f"DataFrame 维度: {btc_df.shape}")
btc_df.tail()

In [ ]:
# ── 专业K线图 ──

def plot_candlestick(df: pd.DataFrame, title: str = "BTC/USDT", last_n: int = 120):
    """绘制带成交量柱的 OHLCV K线图。"""
    data = df.tail(last_n).copy()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[3, 1], sharex=True)
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # K线实体颜色
    colors = ['#26a69a' if c >= o else '#ef5350' for o, c in zip(data['open'], data['close'])]
    
    x = range(len(data))
    
    # 影线（最高-最低线）
    for i, (idx, row) in enumerate(data.iterrows()):
        color = colors[i]
        ax1.plot([i, i], [row['low'], row['high']], color=color, linewidth=0.8)
    
    # 实体
    body_width = 0.6
    for i, (idx, row) in enumerate(data.iterrows()):
        color = colors[i]
        bottom = min(row['open'], row['close'])
        height = abs(row['close'] - row['open'])
        ax1.bar(i, height, bottom=bottom, width=body_width, color=color, edgecolor=color)
    
    ax1.set_ylabel('价格 (USDT)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # 成交量柱
    ax2.bar(x, data['volume'], color=colors, alpha=0.7, width=body_width)
    ax2.set_ylabel('成交量 (BTC)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # X 轴标签（每 N 个时间戳显示一个）
    step = max(1, len(data) // 10)
    ax2.set_xticks(range(0, len(data), step))
    ax2.set_xticklabels(
        [data.index[i].strftime('%m/%d %H:%M') for i in range(0, len(data), step)],
        rotation=45, ha='right'
    )
    
    plt.tight_layout()
    plt.show()

plot_candlestick(btc_df, "BTC/USDT — 小时K线（最近5天）", last_n=120)

---

## 💻 第五节：将数据存储到 SQLite

每次都从 Binance 获取数据既**慢**又会触发频率限制。生产代码将数据缓存在 **SQLite 数据库**中 — 一个轻量级的文件型数据库，非常适合单用户应用。

### 为什么选择 SQLite？

- **零配置**：不需要服务器和凭据 — 只是一个文件
- **ACID 事务**：即使进程崩溃也能保证数据完整性
- **快速查询**：索引查找只需微秒
- **生产验证**：实际交易机器人中用于K线缓存

让我们构建生产版 `BinanceHistoryStore` 的简化版本：

In [ ]:
# ── 从零构建 SQLite K线存储 ──
import sqlite3
import tempfile

class KlineStore:
    """基于 SQLite 的历史K线数据存储。
    
    遵循生产版 BinanceHistoryStore 的模式：
    - Upsert（插入或更新）处理重复获取
    - 复合主键: (symbol, interval, open_time_ms)
    - 索引加速时间范围查询
    """
    
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._initialize()
    
    def _initialize(self):
        """创建K线表和索引（如果不存在）。"""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS klines (
                    symbol TEXT NOT NULL,
                    interval TEXT NOT NULL,
                    open_time_ms INTEGER NOT NULL,
                    close_time_ms INTEGER NOT NULL,
                    open REAL NOT NULL,
                    high REAL NOT NULL,
                    low REAL NOT NULL,
                    close REAL NOT NULL,
                    volume REAL NOT NULL,
                    quote_volume REAL NOT NULL,
                    trade_count INTEGER NOT NULL,
                    PRIMARY KEY (symbol, interval, open_time_ms)
                )
            """)
            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_klines_lookup
                ON klines (interval, symbol, open_time_ms)
            """)
    
    def upsert(self, klines: list[Kline]) -> int:
        """插入K线数据，冲突时更新已有行。"""
        if not klines:
            return 0
        with sqlite3.connect(self.db_path) as conn:
            conn.executemany("""
                INSERT INTO klines (symbol, interval, open_time_ms, close_time_ms,
                                    open, high, low, close, volume, quote_volume, trade_count)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(symbol, interval, open_time_ms) DO UPDATE SET
                    close = excluded.close,
                    high = excluded.high,
                    low = excluded.low,
                    volume = excluded.volume,
                    quote_volume = excluded.quote_volume
            """, [
                (k.symbol, k.interval, k.open_time_ms, k.close_time_ms,
                 k.open, k.high, k.low, k.close, k.volume, k.quote_volume, k.trade_count)
                for k in klines
            ])
        return len(klines)
    
    def query(self, symbol: str, interval: str, limit: int = 500) -> pd.DataFrame:
        """从数据库加载最近的K线为 DataFrame。"""
        with sqlite3.connect(self.db_path) as conn:
            df = pd.read_sql_query(
                "SELECT * FROM klines WHERE symbol = ? AND interval = ? ORDER BY open_time_ms DESC LIMIT ?",
                conn, params=(symbol, interval, limit)
            )
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['open_time_ms'], unit='ms', utc=True)
            df = df.set_index('timestamp').sort_index()
        return df
    
    def count(self, symbol: str, interval: str) -> int:
        """统计指定交易对/间隔的存储K线数量。"""
        with sqlite3.connect(self.db_path) as conn:
            result = conn.execute(
                "SELECT COUNT(*) FROM klines WHERE symbol = ? AND interval = ?",
                (symbol, interval)
            ).fetchone()
        return result[0]

# 存储之前获取的数据
db_path = Path(tempfile.gettempdir()) / "quant_course_klines.db"
store = KlineStore(str(db_path))

stored = store.upsert(btc_klines)
print(f"✅ 已存储 {stored} 根K线到 {db_path}")
print(f"   数据库中有 {store.count('BTCUSDT', '1h')} 根 BTCUSDT 小时K线")

In [ ]:
# ── 从存储中查询数据 ──
cached_df = store.query("BTCUSDT", "1h", limit=100)
print(f"从 SQLite 检索了 {len(cached_df)} 根K线")
cached_df.tail(3)

### 💻 与生产代码存储对比

`bot/data/binance_history_store.py` 中的生产版 `BinanceHistoryStore` 遵循完全相同的模式：

- 相同的 `ON CONFLICT ... DO UPDATE` upsert 策略
- 相同的复合主键 `(symbol, interval, open_time_ms)`
- 额外的 `get_time_range()` 方法检查已缓存的范围
- 使用 `sqlite3.Row` 工厂实现字典式行访问

---

## 🔬 第六节：互动练习

### 练习 1：获取 ETH 数据 🔬

获取 30 天的 ETHUSDT 小时K线，存入同一个数据库，并绘制K线图。

In [ ]:
# ── 练习 1：在这里写代码 ──
# 1. 调用 fetch_all_klines("ETHUSDT", "1h", days=30)
# 2. 用 store.upsert() 存储结果
# 3. 查询并用 plot_candlestick() 绘图

# 你的代码

In [ ]:
# ── 练习 1：参考答案（展开查看）──
# eth_klines = fetch_all_klines("ETHUSDT", "1h", days=30)
# store.upsert(eth_klines)
# eth_df = klines_to_dataframe(eth_klines)
# plot_candlestick(eth_df, "ETH/USDT — 小时K线", last_n=120)

### 练习 2：日线 vs 小时线 ⭐

获取 90 天的 **日线** (`1d`) BTCUSDT K线数据。将日线图与小时图对比。不同的时间间隔如何影响你能观察到的价格模式？

In [ ]:
# ── 练习 2：在这里写代码 ──

# 你的代码

### 练习 3：成交量分析 ⭐

绘制**报价资产成交量**（USDT 交易额）随时间的变化。你能发现异常高成交量的日期吗？那些天发生了什么？

In [ ]:
# ── 练习 3：在这里写代码 ──

# 你的代码

---

## ✅ 知识检查

1. OHLCV K线的五个字段是什么？
2. 为什么 Binance API 以字符串而非数字返回价格？
3. 每次 API 请求最多返回多少根K线？
4. 为什么使用 `ON CONFLICT ... DO UPDATE` 而不是简单的 `INSERT`？
5. 数据类中 `frozen=True` 有什么作用？

<details>
<summary>点击查看答案</summary>

1. 开盘价(Open)、最高价(High)、最低价(Low)、收盘价(Close)、成交量(Volume)
2. 避免浮点精度损失 — 字符串保留精确的十进制表示
3. 每次请求最多 1000 根K线
4. 优雅处理重复获取 — 如果重新获取相同时间范围，已有行会被更新而不是报错
5. 使实例不可变 — 创建后不能意外修改K线记录，防止数据损坏 bug

</details>

---

## 🔗 下一步：Notebook 02 — 从零构建技术指标

现在你已经有了原始价格数据，下一步是从中**提取有意义的信号**。在 Notebook 02 中，你将使用纯 pandas 从零构建 EMA、RSI、布林带、MACD 和波动率 — 这正是驱动我们交易机器人信号引擎的技术指标。

**打开：** `02_从零构建技术指标.ipynb`